# 🎬 EchoVision: Decoupled Audio-Visual RAG Pipeline (Google Colab T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
&nbsp;[![Hardware: NVIDIA Tesla T4 (15GB)](https://img.shields.io/badge/Hardware-NVIDIA%20Tesla%20T4%20(15GB)-green.svg)](https://cloud.google.com)
&nbsp;[![Framework: EchoVision Online](https://img.shields.io/badge/Pipeline-Decoupled%20Audio--Visual%20RAG-blue.svg)](https://github.com)
&nbsp;[![Model: Qwen2.5-VL-3B-Instruct](https://img.shields.io/badge/VLM-Qwen2.5--VL--3B--Instruct-purple.svg)](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)

**EchoVision** is an advanced, decoupled multimodal **Retrieval-Augmented Generation (RAG)** framework for **Video Question Answering (Video QA)**.
This notebook is specifically configured and optimized for **Google Colab's NVIDIA Tesla T4 GPU (15GB VRAM)** to run the **Online Framework (Stage 2 Decoupled Retrieval & Re-ranking + Stage 3 Grounded Generation)** with native PyTorch inference and **zero Ollama server requirement**.

---

### 🏗️ Online Framework Architecture
1. **Question Classification & Modality Estimator ($\beta(q)$)**: Uses `all-MiniLM-L6-v2` to dynamically balance query attention between acoustic audio events and visual keyframes.
2. **Decoupled Multimodal Retrieval**: Independent parallel retrieval across spoken dialogue & acoustic sound events (`LAION-CLAP` / `ChromaDB`) and visual keyframes (`FAISS` / `CLIP` / `Qwen-VL`).
3. **Cross-Encoder Re-Ranking**: Deep cross-attention relevance scoring with `BAAI/bge-reranker-large` incorporating modality bonus ($\beta(q) \cdot \text{Bonus}$) and temporal agreement scoring.
4. **Temporal NMS Deduplication & Sufficiency Gate**: Suppresses overlapping temporal chunks (>80% overlap) and verifies evidence completeness with dynamic loopback fallback.
5. **Stage 3 Grounded Answer Generation**: Grounded generation using native `Qwen/Qwen2.5-VL-3B-Instruct` in FP16 precision, strictly constrained to produce concise, benchmark-ready answers (1–5 words).

---

## 1. 🚀 Step 1: Hardware & T4 GPU Verification
Run the cell below to confirm that Google Colab is running with an **NVIDIA Tesla T4 GPU** (or compatible CUDA accelerator with >= 15GB VRAM).

> 💡 **Tip:** If GPU is not enabled, go to **Runtime > Change runtime type > Hardware accelerator > T4 GPU** and restart the session.

In [ ]:
# Check NVIDIA GPU hardware details and VRAM status
!nvidia-smi

import os
import torch

print("=" * 50)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Model       : {device_name}")
    print(f"Dedicated VRAM  : {total_vram_gb:.2f} GB")
    print(f"Device Index    : cuda:0")
    print("=" * 50)
    print("✅ Hardware verified: NVIDIA CUDA GPU is ready for EchoVision!")
else:
    print("=" * 50)
    print("⚠️ WARNING: CUDA is not active! Please navigate to Runtime -> Change runtime type -> Select T4 GPU.")


## 2. 📦 Step 2: Clone Your GitHub Repository
Enter your GitHub repository URL where you deployed the EchoVision code.
The notebook will automatically clone the repository (or pull latest changes if already present) and set up the working directory.

In [ ]:
#@title 🔗 GitHub Repository Deployment Settings { run: "auto" }
import os
import sys
import shutil

# Insert your GitHub repository URL here
GITHUB_REPO_URL = "https://github.com/G-shubham18/Cloud_EVL2.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
PROJECT_FOLDER = "Cloud_EVL2" #@param {type:"string"}

TARGET_DIR = os.path.abspath(PROJECT_FOLDER)

# Check if we are already in the repository root (e.g. main.py exists)
if os.path.exists("main.py") and os.path.exists("config.py"):
    print(f"✅ Already inside project directory: {os.getcwd()}")
    TARGET_DIR = os.getcwd()
else:
    if os.path.exists(TARGET_DIR) and os.path.exists(os.path.join(TARGET_DIR, ".git")):
        print(f"📁 Directory '{TARGET_DIR}' already exists. Pulling latest updates from branch '{BRANCH}'...")
        !git -C "{TARGET_DIR}" pull origin {BRANCH}
    else:
        if GITHUB_REPO_URL and "your-username" not in GITHUB_REPO_URL:
            print(f"🚀 Cloning repository from {GITHUB_REPO_URL} (branch: {BRANCH})...")
            !git clone -b {BRANCH} {GITHUB_REPO_URL} "{TARGET_DIR}"
        else:
            print("ℹ️ Note: GITHUB_REPO_URL contains default placeholder. Assuming code is already placed in current directory or creating local workspace...")
            os.makedirs(TARGET_DIR, exist_ok=True)

    # Switch working directory to repository root
    if os.path.exists(TARGET_DIR) and os.getcwd() != TARGET_DIR:
        os.chdir(TARGET_DIR)
        print(f"📂 Switched current working directory to: {os.getcwd()}")

# Ensure project directory is at the top of sys.path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
print(f"✅ Workspace configured: {current_dir}")


## 3. 🛠️ Step 3: Install Required Dependencies
Install system packages (FFmpeg) and all required Python packages for EchoVision.

All model dependencies (`Qwen2.5-VL-3B-Instruct`, `bge-reranker-large`, `all-MiniLM-L6-v2`, `faster-whisper`, `laion/clap-htsat-unfused`) will execute natively on the T4 GPU in PyTorch.

In [ ]:
# 1. Install FFmpeg multimedia binary
!apt-get update -qq && apt-get install -y -qq ffmpeg

# 2. If requirements.txt exists, install from it, otherwise install exact required packages
if os.path.exists("requirements.txt"):
    print("Installing dependencies from requirements.txt...")
    !pip install -q -r requirements.txt

# Ensure all specific online packages and utilities are installed
!pip install -q \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    "sentence-transformers" \
    "chromadb" \
    "faiss-cpu" \
    "faster-whisper" \
    "scenedetect" \
    "qwen-vl-utils" \
    "librosa" \
    "imageio-ffmpeg" \
    "opencv-python" \
    "gradio>=4.0.0" \
    "pandas"

print("\n✅ All Python and system dependencies successfully installed!")


## 4. ⚙️ Step 4: Verify Framework Configuration for Colab T4 (FP16 Mode)
Verify that `config.py` correctly detects the T4 GPU:
- **Device**: `cuda`
- **Torch Precision**: `torch.float16` (ideal for NVIDIA Tesla T4 Turing architecture)
- **Stage 3 Generator**: Native `Qwen/Qwen2.5-VL-3B-Instruct`
- **Stage 2 Re-Ranker**: `BAAI/bge-reranker-large`

In [ ]:
import os
import torch

# Ensure CUDA is targeted if available
if torch.cuda.is_available():
    os.environ["FORCE_DEVICE"] = "cuda"

import config

print("=" * 60)
print("             ECHOVISION CONFIGURATION CHECK              ")
print("=" * 60)
print(f"Target Device       : {config.DEVICE}")
print(f"Torch DType         : {config.TORCH_DTYPE}")
print(f"Is GPU Accelerated  : {config.IS_GPU}")
print(f"Stage 2 Reranker    : {config.RERANKER_MODEL}")
print(f"Modality Estimator  : {config.MODALITY_ESTIMATOR_MODEL}")
print(f"Stage 3 Generator   : {config.GENERATOR_MODEL}")
print(f"Vector Store Root   : {os.path.abspath(config.VECTOR_STORE_DIR)}")
print("=" * 60)

if torch.cuda.is_available():
    allocated_mb = torch.cuda.memory_allocated(0) / (1024 ** 2)
    reserved_mb = torch.cuda.memory_reserved(0) / (1024 ** 2)
    print(f"Current VRAM Usage  : Allocated={allocated_mb:.2f} MB, Reserved={reserved_mb:.2f} MB")
    print("\n💡 T4 GPU VRAM Budget Note:")
    print("  - Qwen2.5-VL-3B (FP16) : ~6.5 GB")
    print("  - BGE-Reranker-Large   : ~1.5 GB")
    print("  - All-MiniLM-L6-v2     : ~0.2 GB")
    print("  - Total Online Footprint: ~8.2 GB (Comfortably fits in 15GB T4 VRAM!)")


## 5. 📂 Step 5: Preparing Dataset & Stage 1 Vector Stores
The **Online Framework (Stage 2 & 3)** operates on isolated vector stores located in `data/vector_stores/<video_stem>_<hash>/`:
- **Option A (Google Drive)**: Mount Google Drive if you saved your pre-indexed `data/vector_stores/` there.
- **Option B (Local / Uploaded Zip)**: Unpack pre-indexed vector stores.
- **Option C (Run Ingestion)**: Run Stage 1 Ingestion directly inside Colab on raw videos if vector stores are not yet created.

In [ ]:
#@title 📁 Vector Stores & Dataset Setup { run: "auto" }
import os
import shutil

MOUNT_GOOGLE_DRIVE = False #@param {type:"boolean"}
DRIVE_STORE_PATH = "/content/drive/MyDrive/EchoVision/data/vector_stores" #@param {type:"string"}

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(DRIVE_STORE_PATH):
        os.makedirs("data/vector_stores", exist_ok=True)
        print(f"Syncing pre-indexed vector stores from: {DRIVE_STORE_PATH}...")
        !cp -r "{DRIVE_STORE_PATH}"/* data/vector_stores/
        print("✅ Vector stores synced successfully from Google Drive!")
    else:
        print(f"ℹ️ Google Drive path '{DRIVE_STORE_PATH}' does not exist. Using local data directory.")

# Ensure project directories exist
os.makedirs("data/vector_stores", exist_ok=True)
os.makedirs("data/keyframes", exist_ok=True)
os.makedirs("output", exist_ok=True)

# List available vector stores on disk
available_stores = [d for d in os.listdir("data/vector_stores") if os.path.isdir(os.path.join("data/vector_stores", d))]
print(f"📊 Total Pre-Indexed Video Stores on disk: {len(available_stores)}")
for s in available_stores[:15]:
    print(f"  📁 data/vector_stores/{s}")


### ⚡ (Optional) Run Stage 1 Ingestion on New Videos
If you uploaded new video files (e.g., in `smoketest/videos/` or `dataset/musicAVQA/selected_videos/`) and need to extract features and build vector stores, run this cell.
*If you already have indexed vector stores in `data/vector_stores/`, you can **skip** this step!*

In [ ]:
#@title ⚡ Run Stage 1 Ingestion (Optional) { run: "auto" }
RUN_STAGE1_INGESTION = False #@param {type:"boolean"}
TARGET_INGESTION_DIR = "smoketest" #@param ["smoketest", "dataset/musicAVQA", "dataset/activininetqa", "dataset/moviechat", "dataset"] {allow-input: true}
FORCE_REINDEX = False #@param {type:"boolean"}

if RUN_STAGE1_INGESTION:
    reindex_arg = "--force-reindex" if FORCE_REINDEX else ""
    print(f"Running Stage 1 Ingestion on '{TARGET_INGESTION_DIR}'...")
    !python ingestion.py --dataset_dir {TARGET_INGESTION_DIR} {reindex_arg}
else:
    print("Skipping Stage 1 Ingestion (relying on existing vector stores).")


## 6. 🌐 Step 6: Run Batch Online Framework Evaluation (`main.py`)
Run the batch online question-answering framework across your benchmark dataset:
1. Loads pre-indexed Stage 1 vector stores from disk with zero feature-extraction delay.
2. Reads question JSON files from `<dataset_dir>/json/`.
3. Performs dynamic modality estimation ($\beta(q)$), decoupled audio/visual retrieval, and BGE re-ranking.
4. Synthesizes grounded answers using `Qwen2.5-VL-3B-Instruct`.
5. Writes formatted output files (containing `video_id`, `question`, `predicted_answer`, `ground_truth_answer`, `video_duration_sec`) into `output/`.

In [ ]:
#@title 🚀 Execute Batch Online Evaluation { run: "auto" }
DATASET_PATH = "smoketest" #@param ["smoketest", "dataset/musicAVQA", "dataset/activininetqa", "dataset/moviechat", "dataset"] {allow-input: true}
OUTPUT_DIR = "output" #@param {type:"string"}
AUTO_INGEST_MISSING = True #@param {type:"boolean"}

auto_flag = "--auto-ingest" if AUTO_INGEST_MISSING else ""

print(f"Executing EchoVision Online Framework on: {DATASET_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print("-" * 50)

!python main.py --dataset_dir "{DATASET_PATH}" --output_dir "{OUTPUT_DIR}" {auto_flag}


## 7. 🧠 Step 7: Interactive Python Online QA Engine
Pre-load all Stage 2 & 3 online models once in GPU VRAM and run real-time interactive queries inside the notebook.

The `ask_video(video_path, question)` function executes the full pipeline:
- **Modality classification**: $\beta(q) \in [0, 1]$
- **Decoupled retrieval**: ChromaDB (audio chunks) + FAISS (visual keyframes)
- **Cross-Encoder Re-Ranking**: `bge-reranker-large` with audio bonus & temporal agreement
- **Temporal NMS & Sufficiency verification**
- **Grounded generation**: `Qwen2.5-VL-3B-Instruct`

In [ ]:
import os
import torch
from typing import Dict, Any

from config import KA_DEFAULT, KV_DEFAULT, VECTOR_STORE_DIR
from ingestion import get_video_store_dir, discover_videos
from stage1_offline.vector_indexer import VectorIndexer
from stage2_online.question_classifier import QuestionClassifier
from stage2_online.decoupled_retriever import DecoupledRetriever
from stage2_online.deduplicator import Deduplicator
from stage2_online.reranker import ReRanker
from stage2_online.sufficiency_gate import SufficiencyGate
from stage3_generator.generator import Generator

print("⏳ Pre-loading Stage 2 & Stage 3 online models on GPU...")

# Singleton model instances shared across queries
shared_online_components = {
    'qc': QuestionClassifier(),
    'dedup': Deduplicator(),
    'reranker': ReRanker(),
    'gate': SufficiencyGate(),
    'generator': Generator()
}

print("✅ All Stage 2 & Stage 3 models are loaded and ready in GPU VRAM!")

def ask_video(video_path: str, question: str, verbose: bool = True) -> Dict[str, Any]:
    """
    Runs the real-time online QA pipeline on any video (auto-indexes if needed).
    """
    abs_vpath = os.path.abspath(video_path)
    store_dir = get_video_store_dir(abs_vpath)
    indexer = VectorIndexer(store_dir=store_dir)

    # Auto-ingest if not already indexed
    if not indexer.is_indexed(abs_vpath):
        print(f"ℹ️ Stage 1 index missing for '{os.path.basename(abs_vpath)}'. Running automatic Stage 1 Ingestion...")
        from ingestion import Stage1Ingestor
        ingestor = Stage1Ingestor()
        indexer, _, err = ingestor.process_single_video(abs_vpath)
        if indexer is None:
            raise RuntimeError(f"Failed to ingest video '{abs_vpath}': {err}")

    retriever = DecoupledRetriever(indexer=indexer)
    qc = shared_online_components['qc']
    dedup = shared_online_components['dedup']
    reranker = shared_online_components['reranker']
    gate = shared_online_components['gate']
    generator = shared_online_components['generator']

    # 1. Modality Estimation
    beta_q = qc.estimate_beta(question)
    total_k = KA_DEFAULT + KV_DEFAULT
    k_a = max(2, int(round(total_k * beta_q)))
    k_v = max(2, int(round(total_k * (1.0 - beta_q))))

    # 2. Decoupled Retrieval
    audio_cands, visual_cands = retriever.retrieve(question, k_a=k_a, k_v=k_v)
    combined = audio_cands + visual_cands
    if not combined:
        fallback_vis = retriever.retrieve_visual(question, k=10)
        combined = fallback_vis if fallback_vis else []

    # 3. Cross-Encoder Re-Ranking
    scored = reranker.score_candidates(question, combined, beta_q)

    # 4. Temporal NMS Deduplication
    dedup_cands = dedup.apply_nms(scored, score_key="final_score")
    top_cands = dedup_cands[:total_k]

    # 5. Grounded Generation
    context = generator.format_context(top_cands)
    answer = generator.generate_answer(question, context)

    if verbose:
        print("\n" + "=" * 65)
        print(f"🎬 Video       : {os.path.basename(abs_vpath)}")
        print(f"❓ Question    : {question}")
        print(f"⚖️ Beta(q)     : {beta_q:.2f} (Audio: {beta_q * 100:.1f}%, Visual: {(1.0 - beta_q) * 100:.1f}%)")
        print(f"🎯 Prediction  : {answer}")
        print("-" * 65)
        print("📑 Top Grounded Evidence Chunks:")
        for idx, c in enumerate(top_cands[:3], start=1):
            meta = c.get('metadata', {})
            st = meta.get('start_time', 0.0)
            et = meta.get('end_time', 0.0)
            mtype = str(meta.get('type', 'visual')).upper()
            score = c.get('final_score', 0.0)
            snippet = c.get('text', '').strip().replace('\n', ' ')
            print(f"  [{idx}] [{mtype} | {st:.1f}s - {et:.1f}s | Score: {score:.3f}]")
            print(f"      \"{snippet[:140]}...\"")
        print("=" * 65 + "\n")

    return {
        "video": abs_vpath,
        "question": question,
        "answer": answer,
        "beta_q": beta_q,
        "evidence": top_cands
    }


### 💬 Run an Interactive Query:
Select any video from your dataset and enter a question below:

In [ ]:
#@title 🔍 Interactive Video QA Query { run: "auto" }
# Discover candidate videos automatically
candidate_dirs = ["smoketest/videos", "dataset/videos", "dataset/musicAVQA/selected_videos", "dataset/activininetqa/videos", "dataset/moviechat/videos"]
discovered = []
for cd in candidate_dirs:
    if os.path.exists(cd):
        discovered.extend(discover_videos(cd))

if not discovered:
    print("⚠️ No video files discovered in standard paths. Using placeholder path.")
    discovered = ["smoketest/videos/10.mp4"]

TEST_VIDEO_PATH = discovered[0] #@param {type:"string"}
USER_QUESTION = "What is happening in this video?" #@param {type:"string"}

if os.path.exists(TEST_VIDEO_PATH):
    res = ask_video(TEST_VIDEO_PATH, USER_QUESTION)
else:
    print(f"File not found: {TEST_VIDEO_PATH}. Please provide a valid video path.")


## 8. 🖥️ Step 8: Interactive Web UI (Gradio on Google Colab)
Run a full web user interface directly inside Colab (or through a public shareable URL) for visual interaction, real-time testing, and demonstrations.

In [ ]:
import gradio as gr
import os

# Collect all indexed video options
def get_available_videos():
    v_list = []
    for d in ["smoketest/videos", "dataset/videos", "dataset/musicAVQA/selected_videos", "dataset/activininetqa/videos", "dataset/moviechat/videos"]:
        if os.path.exists(d):
            v_list.extend(discover_videos(d))
    return sorted(list(set(v_list)))

video_options = get_available_videos()
if not video_options:
    video_options = ["No videos found (Please ingest videos first)"]

def gradio_handler(video_selection, query_text):
    if not video_selection or video_selection.startswith("No videos found"):
        return "⚠️ Error: No valid video selected.", "N/A", "Please ingest or upload videos first."
    if not query_text or not query_text.strip():
        return "⚠️ Please enter a question.", "N/A", ""

    try:
        result = ask_video(video_selection, query_text, verbose=False)
        ans = result["answer"]
        beta = result["beta_q"]
        modality_text = f"Audio Weight: {beta * 100:.1f}% | Visual Weight: {(1.0 - beta) * 100:.1f}%"

        ev_md = []
        for i, c in enumerate(result["evidence"][:4], start=1):
            meta = c.get('metadata', {})
            st = meta.get('start_time', 0.0)
            et = meta.get('end_time', 0.0)
            mtype = str(meta.get('type', 'visual')).capitalize()
            score = c.get('final_score', 0.0)
            text = c.get('text', '').strip()
            ev_md.append(f"**[{i}] {mtype} ({st:.1f}s - {et:.1f}s) | Score: {score:.2f}**\n> {text}\n")
        
        return ans, modality_text, "\n".join(ev_md)
    except Exception as e:
        return f"Error: {str(e)}", "Error", ""

# Construct modern Gradio interface
with gr.Blocks(title="EchoVision Online Video QA", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 EchoVision: Online Video QA Framework")
    gr.Markdown("Decoupled Audio-Visual Retrieval-Augmented Generation running on **Google Colab NVIDIA T4 GPU**.")
    
    with gr.Row():
        with gr.Column(scale=1):
            v_dropdown = gr.Dropdown(choices=video_options, value=video_options[0], label="Select Video")
            q_input = gr.Textbox(placeholder="e.g. What color is the person's shirt? Or what sound is playing?", label="Question", lines=2)
            run_btn = gr.Button("🔍 Run Online QA", variant="primary")
            
        with gr.Column(scale=1):
            ans_output = gr.Textbox(label="🎯 Predicted Grounded Answer", lines=2)
            mod_output = gr.Textbox(label="⚖️ Dynamic Modality Weight beta(q)")
            ev_output = gr.Markdown(label="📑 Top Retrieved Audio-Visual Evidence")
            
    run_btn.click(fn=gradio_handler, inputs=[v_dropdown, q_input], outputs=[ans_output, mod_output, ev_output])

# Launch with public shareable link
demo.launch(share=True, debug=False)


## 9. 📊 Step 9: Inspect, Evaluate & Download Results
Examine the predictions generated by the batch pipeline, calculate accuracy (Exact Match / Substring Match against Ground Truth), and display results in a structured table.

In [ ]:
import os
import json
import pandas as pd

output_dir = "output"
if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

json_files = [f for f in os.listdir(output_dir) if f.endswith(".json") and not f.startswith("ingestion_latency")]
print(f"📁 Output result JSON files found: {json_files}")

all_rows = []
for jf in json_files:
    fpath = os.path.join(output_dir, jf)
    try:
        with open(fpath, "r", encoding="utf-8") as f:
            entries = json.load(f)
            if isinstance(entries, list):
                for item in entries:
                    item["source_file"] = jf
                    all_rows.append(item)
    except Exception as e:
        print(f"Error reading {fpath}: {e}")

if all_rows:
    df = pd.DataFrame(all_rows)
    print(f"\n📊 Total Questions Processed: {len(df)}")

    # Benchmark Evaluation
    if "ground_truth_answer" in df.columns and "predicted_answer" in df.columns:
        def check_correct(row):
            gt = str(row["ground_truth_answer"]).strip().lower()
            pred = str(row["predicted_answer"]).strip().lower()
            if not gt:
                return None
            return (gt == pred) or (gt in pred) or (pred in gt)

        df["is_correct"] = df.apply(check_correct, axis=1)
        eval_subset = df.dropna(subset=["is_correct"])
        if len(eval_subset) > 0:
            acc = eval_subset["is_correct"].mean() * 100
            print(f"🎯 Accuracy Score: {acc:.2f}% ({int(eval_subset['is_correct'].sum())}/{len(eval_subset)})")

    display_cols = [c for c in ["video_id", "question", "predicted_answer", "ground_truth_answer", "video_duration_sec"] if c in df.columns]
    try:
        from IPython.display import display
        display(df[display_cols].head(25))
    except Exception:
        print(df[display_cols].head(25).to_string())
else:
    print("ℹ️ No output JSON files found yet. Run Step 6 to generate predictions.")


## 10. 🧹 Step 10: VRAM Memory Cleanup & Cache Management
Free cached GPU memory after completing evaluations to maintain peak performance.

In [ ]:
import gc
import torch

def release_gpu_resources():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        allocated = torch.cuda.memory_allocated(0) / (1024 ** 2)
        reserved = torch.cuda.memory_reserved(0) / (1024 ** 2)
        print(f"🧹 GPU VRAM Cleared! Allocated: {allocated:.2f} MB | Reserved: {reserved:.2f} MB")

release_gpu_resources()
